# Hamiltonian Eigendecomposition — Pan-Cancer Multiomics

**Mayfield JD.** *Phase Transitions in Pan-Cancer Multiomics Covariance Landscapes Define a Universal Malignancy Coordinate.* Bioinformatics, 2026.

This notebook demonstrates the core algorithm:
1. Preprocessing and multimodal alignment
2. Hamiltonian construction (H = X^T X / n)
3. Eigendecomposition and survival-ranked mode selection
4. Spectral concentration and universal malignancy coordinate η
5. Free energy landscape and critical temperature T*

**Data:** Replace the placeholder data-loading section (Cell 2) with your own
multiomics DataFrames. CPTAC data are available via the
[cptac Python package](https://github.com/PayneLab/cptac);
TCGA data via [cBioPortal](https://www.cbioportal.org).


## 1. Dependencies

In [ ]:
# Install dependencies (run once)
# pip install numpy scipy scikit-learn lifelines pandas matplotlib seaborn


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

from src.eigendecomposition import (
    preprocess_modality,
    align_modalities,
    build_hamiltonian,
    decompose,
    compute_eigenscores,
    spectral_concentration,
    rank_modes,
    compute_eigenindex,
    compute_eta,
    run_eigendecomposition,
)
from src.phase_transition import run_phase_transition

print("Imports complete.")


## 2. Data Loading

Replace this section with your own data source.
Each modality should be a pandas DataFrame with:
- **Rows** = samples (indexed by sample/patient ID)
- **Columns** = features (genes, proteins, CNV segments, etc.)

Survival arrays `os_time` and `os_event` must be aligned to the same sample
order as the modality DataFrames.


In [ ]:
# ── REPLACE THIS CELL WITH YOUR DATA LOADING CODE ────────────────────────────
#
# Example expected format:
#
#   prot  = pd.DataFrame(...)   # samples × protein features
#   trans = pd.DataFrame(...)   # samples × transcript features
#   cnv   = pd.DataFrame(...)   # samples × CNV features
#
#   os_time  = np.array(...)    # overall survival time (days), shape (n_samples,)
#   os_event = np.array(...)    # event indicator (1=event, 0=censored)
#
# ─────────────────────────────────────────────────────────────────────────────

raise NotImplementedError(
    "Replace this cell with your data loading code. "
    "See repository README.md for data access instructions."
)


## 3. Preprocessing

In [ ]:
# Preprocess each modality: drop high-missing features, impute, variance-select
# Modality prefix is added so features can be traced back after concatenation

prot_clean  = preprocess_modality(prot,  modality_prefix="proteomics")
trans_clean = preprocess_modality(trans, modality_prefix="transcriptomics")
cnv_clean   = preprocess_modality(cnv,   modality_prefix="cnv")

print(f"Proteomics features:      {prot_clean.shape[1]}")
print(f"Transcriptomics features: {trans_clean.shape[1]}")
print(f"CNV features:             {cnv_clean.shape[1]}")


In [ ]:
# Align modalities — retain only samples present in all three
# (complete-data alignment across proteomics, transcriptomics, and CNV)

modality_dict = {
    "proteomics":      prot_clean,
    "transcriptomics": trans_clean,
    "cnv":             cnv_clean,
}

X_df = align_modalities(modality_dict)
print(f"Aligned matrix: {X_df.shape[0]} samples × {X_df.shape[1]} features")

# Align survival arrays to the aligned sample index
n = X_df.shape[0]
os_time_aligned  = os_time[:n]
os_event_aligned = os_event[:n]
print(f"Events: {int(os_event_aligned.sum())} / {n}")


## 4. Hamiltonian Construction

In [ ]:
# Build the Hamiltonian H = X^T X / n
# The scaler is saved for later projection of new (e.g., TCGA) data

X   = X_df.values.astype(float)
H, scaler = build_hamiltonian(X)

print(f"Hamiltonian shape: {H.shape}")
print(f"Symmetric: {np.allclose(H, H.T)}")
print(f"Feature matrix shape: {X.shape}")


## 5. Eigendecomposition

In [ ]:
# Decompose H into eigenvalues and eigenvectors
# Uses numpy.linalg.eigh (exact, exploits symmetry)
# Returns eigenvalues in descending order; negatives clipped to zero

eigenvalues, eigenvectors = decompose(H, n_modes=30)

print(f"Eigenvalues shape: {eigenvalues.shape}")
print(f"Eigenvectors shape: {eigenvectors.shape}")
print(f"Top 5 eigenvalues: {eigenvalues[:5].round(4)}")


In [ ]:
# Spectral concentration c = lambda_1 / sum(lambda_k)
# Fraction of total variance carried by the dominant eigenmode

c = spectral_concentration(eigenvalues)
print(f"Spectral concentration c = {c:.4f}")
print(f"(Higher c = more variance concentrated in dominant mode)")


In [ ]:
# Compute per-sample eigenscores: s_ik = x_i^T v_k
X_std = scaler.transform(X)
scores = compute_eigenscores(X_std, eigenvectors)
print(f"Eigenscores shape: {scores.shape}  (samples × modes)")


## 6. Survival-Ranked Mode Selection

In [ ]:
# Rank eigenmodes by AUC against a binary survival label
# AUC is direction-corrected: max(AUC, 1-AUC)
# Top 3 modes are used for the composite eigenindex

top_indices, top_aucs = rank_modes(
    scores, os_time_aligned, os_event_aligned, top_k=3
)

print("Top 3 survival-discriminating modes:")
for rank, (idx, auc) in enumerate(zip(top_indices, top_aucs), 1):
    print(f"  Rank {rank}: Mode {idx:3d}  AUC = {auc:.4f}")


In [ ]:
# Composite eigenindex = mean of top-k survival mode eigenscores
eigenindex = compute_eigenindex(scores, top_indices)

# Universal malignancy coordinate eta = (eigenindex - median) / c
eta = compute_eta(eigenindex, c)

print(f"Eigenindex range: [{eigenindex.min():.3f}, {eigenindex.max():.3f}]")
print(f"η range:          [{eta.min():.3f}, {eta.max():.3f}]")


## 7. Free Energy Landscape and Critical Temperature T*

In [ ]:
# Treat eigenvalues as energy levels; compute the statistical-mechanical
# free energy landscape F(T) = -T log Z(T) and heat capacity C_v(T)
# T* is the critical temperature at which C_v peaks (phase transition analog)

pt = run_phase_transition(eigenvalues)

print(f"Critical temperature T* = {pt['T_star']:.4f}")
print(f"C_v peak height         = {pt['peak_height']:.4f}")


In [ ]:
# Plot free energy landscape and heat capacity
fig, ax1 = plt.subplots(figsize=(8, 4))
ax2 = ax1.twinx()

ax1.plot(pt["T_grid"], pt["F"],   color="#2166ac", lw=2.0, label="F(T)")
ax2.plot(pt["T_grid"], pt["C_v"], color="#d6604d", lw=1.6,
         linestyle="--", label="C_v(T)")
ax1.axvline(pt["T_star"], color="black", lw=1.2,
            linestyle=":", label=f"T* = {pt['T_star']:.3f}")

ax1.set_xlabel("Temperature T", fontsize=11)
ax1.set_ylabel("F(T)  Helmholtz free energy", color="#2166ac", fontsize=10)
ax2.set_ylabel("C_v(T)  heat capacity",        color="#d6604d", fontsize=10)
ax1.set_title("Free energy landscape  |  Phase transition at T*", fontsize=11)

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, fontsize=9, loc="upper right")

plt.tight_layout()
plt.show()
print(f"Spectral concentration c = {c:.4f}   T* = {pt['T_star']:.4f}")


## 8. Convenience: Full Pipeline

The `run_eigendecomposition()` function runs all steps above in one call.


In [ ]:
# Single-call alternative to steps 3–6
results = run_eigendecomposition(
    modality_dict    = modality_dict,
    os_time          = os_time,
    os_event         = os_event,
    n_modes          = 30,
    top_k            = 3,
)

print("run_eigendecomposition() output keys:")
for k, v in results.items():
    shape = getattr(v, "shape", None)
    print(f"  {k:20s}: {shape if shape is not None else v}")


## Summary

| Quantity | Symbol | Description |
|---|---|---|
| Hamiltonian | H | Covariance matrix X^T X / n |
| Eigenvalues | λ_k | Energetic scale of each collective mode |
| Eigenvectors | v_k | Collective modes of molecular co-variation |
| Spectral concentration | c | λ_1 / Σλ — variance in dominant mode |
| Eigenindex | s | Mean eigenscore across top survival modes |
| Malignancy coordinate | η | (s − median(s)) / c |
| Critical temperature | T* | Peak of heat capacity C_v(T) |

See `src/eigendecomposition.py` and `src/phase_transition.py` for full
documentation of each function.
